In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/yolo-dataset-ver3/final_4class_dataset")

yaml_text = f"""path: {DATASET_DIR}

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: cat
  2: crow
  3: trash_bag
"""

KAGGLE_YAML = Path("/kaggle/working/data.yaml")
KAGGLE_YAML.write_text(yaml_text, encoding="utf-8")

print(KAGGLE_YAML.read_text())

path: /kaggle/input/yolo-dataset-ver3/final_4class_dataset

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: cat
  2: crow
  3: trash_bag



In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.8 MB/s eta 0:00:00


In [5]:
from pathlib import Path

print("=== data.yaml ===")
for p in Path("/kaggle/input").rglob("data.yaml"):
    print(p)

print("\n=== final_4class_dataset ===")
for p in Path("/kaggle/input").rglob("final_4class_dataset"):
    if p.is_dir():
        print(p)

=== data.yaml ===
/kaggle/input/datasets/dlguswlsid/yolo-dataset-ver3/final_4class_dataset/data.yaml

=== final_4class_dataset ===
/kaggle/input/datasets/dlguswlsid/yolo-dataset-ver3/final_4class_dataset


In [6]:
from pathlib import Path

DATASET_DIR = Path(
    "/kaggle/input/datasets/dlguswlsid/yolo-dataset-ver3/final_4class_dataset"
)

yaml_text = f"""path: {DATASET_DIR}

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: cat
  2: crow
  3: trash_bag
"""

KAGGLE_YAML = Path("/kaggle/working/data.yaml")

KAGGLE_YAML.write_text(
    yaml_text,
    encoding="utf-8"
)

print(KAGGLE_YAML.read_text())

path: /kaggle/input/datasets/dlguswlsid/yolo-dataset-ver3/final_4class_dataset

train: images/train
val: images/val
test: images/test

names:
  0: person
  1: cat
  2: crow
  3: trash_bag



In [7]:
print("train:", (DATASET_DIR / "images/train").exists())
print("val:  ", (DATASET_DIR / "images/val").exists())
print("test: ", (DATASET_DIR / "images/test").exists())

train: True
val:   True
test:  True


In [8]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    patience=20,
    workers=2,
    device=0,
    project="/kaggle/working/runs_final",
    name="person_cat_crow_trash_model_v3",
    exist_ok=False,
    plots=True
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=person_cat_crow_trash_model_v3, nbs=

In [9]:
from ultralytics import YOLO

# 가장 성능이 좋았던 v3 모델
model = YOLO(
    "/kaggle/working/runs_final/"
    "person_cat_crow_trash_model_v3/"
    "weights/best.pt"
)

# ONNX 변환
model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs_final/person_cat_crow_trash_model_v3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 235ms
Prepared 2 packages in 268ms
Installed 2 packages in 12ms
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.95...
ONNX: ex

'/kaggle/working/runs_final/person_cat_crow_trash_model_v3/weights/best.onnx'

In [11]:
import onnx

model = onnx.load(str(onnx_path))
onnx.checker.check_model(model)

print("ONNX 모델 정상")

for inp in model.graph.input:
    print("입력 이름:", inp.name)

    shape = [
        d.dim_value if d.dim_value > 0 else "dynamic"
        for d in inp.type.tensor_type.shape.dim
    ]

    print("입력 Shape:", shape)

for out in model.graph.output:
    print("출력 이름:", out.name)

    shape = [
        d.dim_value if d.dim_value > 0 else "dynamic"
        for d in out.type.tensor_type.shape.dim
    ]

    print("출력 Shape:", shape)

ONNX 모델 정상
입력 이름: images
입력 Shape: [1, 3, 640, 640]
출력 이름: output0
출력 Shape: [1, 8, 8400]
